In [0]:
# ==========================================================
# 04_silver_meta_uf
# Camada: Silver
# Fonte: 2026-11-01_meta_alfabetizacao_uf.parquet (Bronze)
# Destino: default.meta_uf
#
# RESPONSABILIDADE
# Transformar a publicação oficial das metas de alfabetização
# (nível UF) em tabela Silver confiável.
#
# NOTA SOBRE A COLUNA rede
# Nesta tabela rede é texto ('Pública'), não código numérico
# como no indicador. O join com indicador_uf é por sigla_uf
# e ano, não por rede.
#
# NOTA SOBRE O RR
# O RR possui linhas com taxa e metas nulas na origem. Esses
# registros são preservados: a ausência é informação válida
# e a Gold os classifica como 'sem_indicador'.
#
# NOTA DE PERMISSÃO
# O schema silver pertence a outro usuário. A gravação ocorre
# em default, conforme adaptação documentada na seção 10.
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA_DESTINO = "default"
TABELA_DESTINO = "meta_uf"

BASE_BRONZE = (
    "/Workspace/Users/rm374983@fiap.com.br/"
    "postech-2iast-fase3-modelo-preditivo-g28/dados/bronze"
)
ARQUIVO_ORIGEM = "2026-11-01_meta_alfabetizacao_uf.parquet"
CAMINHO_ORIGEM = f"{BASE_BRONZE}/{ARQUIVO_ORIGEM}"

TABELA_FULL = f"{CATALOGO}.{SCHEMA_DESTINO}.{TABELA_DESTINO}"

print(f"Origem : {CAMINHO_ORIGEM}")
print(f"Destino: {TABELA_FULL}")

In [0]:
# ==========================================================
# Célula 2 — Leitura da Bronze
# ==========================================================

df_raw = spark.read.parquet(CAMINHO_ORIGEM)

print(f"Registros lidos: {df_raw.count():,}")
print(f"Colunas: {len(df_raw.columns)}")
df_raw.printSchema()

In [0]:
# ==========================================================
# Célula 3 — Transformação e tipagem
#
# Decisões:
# - taxa_alfabetizacao e percentual_participacao mantidas:
#   a taxa é o valor observado por UF, útil para a Gold.
# - As 7 colunas de meta anual são mantidas como estão.
# - Dedupe por (ano, sigla_uf, rede).
# - Linhas com valores nulos são preservadas (caso RR).
# ==========================================================

df_silver = (
    df_raw
    .select(
        F.col("ano").cast("int").alias("ano"),
        F.col("sigla_uf").cast("string").alias("sigla_uf"),
        F.col("rede").cast("string").alias("rede"),
        F.col("taxa_alfabetizacao").cast("double").alias("taxa_alfabetizacao"),
        *[
            F.col(f"meta_alfabetizacao_{a}").cast("double").alias(f"meta_alfabetizacao_{a}")
            for a in range(2024, 2031)
        ],
        F.col("percentual_participacao").cast("double").alias("percentual_participacao"),
    )
    .filter(F.col("ano").isNotNull() & F.col("sigla_uf").isNotNull())
    .dropDuplicates(["ano", "sigla_uf", "rede"])
)

print(f"Registros após transformação: {df_silver.count():,}")

In [0]:
# ==========================================================
# Célula 4 — Rastreabilidade
# ingested_at, source e version em todos os registros.
# ==========================================================

df_silver = (
    df_silver
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source", F.lit(ARQUIVO_ORIGEM))
    .withColumn("version", F.lit("v2"))
)

df_silver.select(
    "ano", "sigla_uf", "rede", "taxa_alfabetizacao",
    "meta_alfabetizacao_2024", "meta_alfabetizacao_2030",
    "ingested_at", "source", "version"
).show(5, truncate=False)

In [0]:
# ==========================================================
# Célula 5 — Data Quality
# Regras: completude, domínio e unicidade.
#
# ATENÇÃO: taxa nula NÃO é falha crítica aqui. O RR tem
# linhas legitimamente nulas na origem. A regra crítica é
# apenas sobre a chave (ano, sigla_uf, rede).
# ==========================================================

total = df_silver.count()

nulos_uf = df_silver.filter(F.col("sigla_uf").isNull()).count()
nulos_rede = df_silver.filter(F.col("rede").isNull()).count()

# Domínio: 27 UFs brasileiras
UFS_VALIDAS = [
    "AC","AL","AM","AP","BA","CE","DF","ES","GO","MA","MG","MS","MT",
    "PA","PB","PE","PI","PR","RJ","RN","RO","RR","RS","SC","SE","SP","TO"
]
ufs_invalidas = df_silver.filter(~F.col("sigla_uf").isin(UFS_VALIDAS)).count()

# Domínio: metas preenchidas devem estar entre 0 e 100
metas = [f"meta_alfabetizacao_{a}" for a in range(2024, 2031)]
fora_faixa = df_silver.filter(
    F.greatest(*[
        F.when((F.col(m) < 0) | (F.col(m) > 100), 1).otherwise(0) for m in metas
    ]) == 1
).count()

chave = ["ano", "sigla_uf", "rede"]
duplicatas = total - df_silver.dropDuplicates(chave).count()

# Diagnóstico: linhas totalmente nulas (esperado para RR)
linhas_vazias = df_silver.filter(
    F.col("taxa_alfabetizacao").isNull() & F.col("meta_alfabetizacao_2024").isNull()
).count()

print("=" * 55)
print("DATA QUALITY — meta_uf")
print("=" * 55)
print(f"Registros totais     : {total:,}")
print(f"UF nula              : {nulos_uf:,}")
print(f"Rede nula            : {nulos_rede:,}")
print(f"UF fora do domínio   : {ufs_invalidas:,}")
print(f"Meta fora de 0-100   : {fora_faixa:,}")
print(f"Duplicatas na chave  : {duplicatas:,}")
print(f"Linhas sem valores   : {linhas_vazias:,} (esperado: RR)")
print(f"Anos                 : {[r[0] for r in df_silver.select('ano').distinct().orderBy('ano').collect()]}")
print(f"UFs distintas        : {df_silver.select('sigla_uf').distinct().count():,}")
print("=" * 55)

assert nulos_uf == 0, "FALHA CRÍTICA: sigla_uf nula"
assert nulos_rede == 0, "FALHA CRÍTICA: rede nula"
assert ufs_invalidas == 0, "FALHA CRÍTICA: sigla_uf fora do domínio"
assert fora_faixa == 0, "FALHA CRÍTICA: meta fora do domínio 0-100"
assert duplicatas == 0, "FALHA CRÍTICA: duplicatas na chave"
print("Todas as regras críticas passaram.")

In [0]:
# ==========================================================
# Célula 6 — Gravação da Silver
# ==========================================================

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_FULL))

print(f"Gravado em {TABELA_FULL}")
print(f"Registros: {spark.table(TABELA_FULL).count():,}")

In [0]:
# ==========================================================
# Célula 7 — Validação pós-gravação
# ==========================================================

df_check = spark.table(TABELA_FULL)

print(f"Registros na tabela : {df_check.count():,}")
print(f"Colunas             : {len(df_check.columns)}")
print(f"Anos                : {[r[0] for r in df_check.select('ano').distinct().orderBy('ano').collect()]}")
print(f"UFs                 : {df_check.select('sigla_uf').distinct().count():,}")

print("\nUFs com meta preenchida:")
df_check.filter(F.col("meta_alfabetizacao_2024").isNotNull()) \
    .select("sigla_uf").distinct().orderBy("sigla_uf").show(30)

print("\nUFs sem meta (esperado: RR):")
df_check.filter(F.col("meta_alfabetizacao_2024").isNull()) \
    .select("sigla_uf", "ano").distinct().orderBy("sigla_uf", "ano").show()

print("\nAmostra:")
df_check.select(
    "ano", "sigla_uf", "taxa_alfabetizacao",
    "meta_alfabetizacao_2024", "meta_alfabetizacao_2025",
    "percentual_participacao"
).orderBy("sigla_uf", "ano").show(10, truncate=False)